# Impact Fund Name Screener

Identifies funds whose name suggests an impact mandate via regex matching against Dirk's keyword list (English + multilingual equivalents).

**Run:** Kernel → Restart & Run All  
**Edit:** Cell 1 (paths/columns) and Cell 2 (patterns) only.

## CELL 1 — Configuration

Edit paths and column list here. All other cells run without changes.

In [2]:
import os, re, json
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
  
config = {}
with open("File_Directory.txt") as f:
    for line in f:
        if ":" in line:
            key, val = line.split(":", 1)
            config[key.strip()] = val.strip()

INPUT_FILE  = Path(config["Input"])
OUTPUT_DIR  = Path(config["Output"])

NAME_COL    = "Name"
ID_COL      = "FundId"

OBJECTIVE_COLUMNS = [
    "Prospectus Objective",
    "KIID Objective/Investment Policy",
    "PRIIPS KID Objective",
    "Strategy Description",
    "Investment Strategy - English",
    "PRIIPS KID Objective - Danish",
    "PRIIPS KID Objective - Dutch",
    "PRIIPS KID Objective - Finnish",
    "PRIIPS KID Objective - French",
    "PRIIPS KID Objective - German",
    "PRIIPS KID Objective - Italian",
    "PRIIPS KID Objective - Norwegian",
    "PRIIPS KID Objective - Portuguese",
    "PRIIPS KID Objective - Spanish",
    "PRIIPS KID Objective - Swedish",
    "KIID Objective/Investment Policy - German",
    "KIID Objective/Investment Policy - French",
    "KIID Objective/Investment Policy - Italian",
    "KIID Objective/Investment Policy - Spanish",
    "KIID Objective/Investment Policy - Norwegian",
    "KIID Objective/Investment Policy - Swedish",
    "KIID Objective/Investment Policy - Finnish",
    "KIID Objective/Investment Policy - Portuguese",
    "KIID Objective/Investment Policy - Danish",
    "Investment Strategy - Danish",
    "Investment Strategy - Finnish",
    "Investment Strategy - French",
    "Investment Strategy - German",
    "Investment Strategy - Italian",
    "Investment Strategy - Norwegian",
    "Investment Strategy - Portuguese",
    "Investment Strategy - Spanish",
    "Investment Strategy - Swedish",
]

print("Configuration loaded.")
print(f"  Input:  {INPUT_FILE}")
print(f"  Output: {OUTPUT_DIR}")


Configuration loaded.
  Input:  /Users/dannyhogan/Desktop/Hogan_RA_Work/Download Sustainable Funds 2026-04-15.xlsx
  Output: /Users/dannyhogan/Desktop/Hogan_RA_Work


## CELL 2 — Keyword Patterns

One entry per `(label, language, regex, needs_review)`. `needs_review=True` flags the match for human inspection — use for ambiguous abbreviations or high false-positive-risk terms.

In [3]:
# Each entry: (label, language, regex_pattern, needs_review)
#
# needs_review=True  → match is flagged for human inspection
#                       because the abbreviation is ambiguous
#                       or the term has false-positive risk.
# ============================================================

# re.IGNORECASE applied at compile time for all patterns
FLAGS = re.IGNORECASE

PATTERNS = [

    # ── International labels ────────────────────────────────
    # Added after Dirk green-light (2026-06-23).
    ("esg",                     "EN",    r"\bESG\b",                                            False),  # Environmental, Social and Governance (164 funds)
    ("sri",                     "EN",    r"\bSRI\b",                                            False),  # Socially Responsible Investment (45 funds; confirmed in scope)

    # ── English ────────────────────────────────────────────
    ("impact",                  "EN",    r"\bimpact\b",                                         False),
    ("positive_change",         "EN",    r"\bpositive[\s\-]change\b",                           False),
    ("better_world",            "EN",    r"\bbetter[\s\-]world\b",                              False),
    ("better_future",           "EN",    r"\bbetter[\s\-]future\b",                             False),  # NEW
    ("future_generations",      "EN",    r"\bfuture(?:[\s\-]for)?[\s\-]generations?\b",         False),
    ("generations_abbrev",      "EN",    r"\bGens\b",                                           True),   # e.g. "Food For Gens"
    ("sdg",                     "EN",    r"\bSDGs?\b",                                          False),
    ("sustainable_development", "EN",    r"\bsustainable[\s\-]dev(?:elopment)?\b",              False),
    ("sustainable",             "EN",    r"\bsustain(?:able|ability|s|ing)?\b",                 False),  # NEW — standalone: Sustainable, Sustainability (113 funds)
    ("sust_dev_abbrev",         "EN",    r"\bSust(?:[\s\-]Dev\b|\b)",                           True),   # Sust alone is ambiguous
    ("responsible",             "EN",    r"\bresponsib(?:le|ility)\b",                          False),  # NEW (12 funds)
    ("ethical",                 "EN",    r"\bethical?\b|\bethics\b",                             False),  # NEW (12 funds)
    ("transformation",          "EN",    r"\btransform(?:ation|ative|ations)?\b",               False),
    ("transition",              "EN",    r"\btransition\b",                                     False),
    ("trans_abbrev",            "EN",    r"\bTrans\b",                                          True),   # Trans alone is ambiguous
    ("climate",                 "EN",    r"\bclimate\b",                                        False),  # NEW — all forms: Climate, Climate Change, Climate-Aligned (51 funds)
    ("climate_action",          "EN",    r"\bclimate[\s\-]action\b",                            False),  # specific phrase kept for label precision
    ("clean_energy",            "EN",    r"\bclean[\s\-]energ(?:y|ies)\b|\bcleantech\b|\bclean[\s\-]tech\b", False),  # NEW (7 funds)
    ("renewable_energy",        "EN",    r"\brenewable(?:[\s\-]energ(?:y|ies))?\b",             False),  # NEW (3 funds)
    ("net_zero",                "EN",    r"\bnet[\s\-]?zero\b|TargetNetZero",                   False),  # NEW (8 funds)
    ("low_carbon",              "EN",    r"\blow[\s\-]carbon\b",                                False),  # NEW (2 funds)
    ("fossil_free",             "EN",    r"\bfossil[\s\-]free\b|\bnon[\s\-]?fossil\b",          False),  # NEW (2 funds)
    ("paris_aligned",           "EN",    r"\bparis[\s\-]?align",                                False),  # NEW (3 funds)
    ("carbon",                  "EN",    r"\bcarbon\b",                                         False),  # NEW — standalone Carbon (9 funds)
    ("environmental",           "EN",    r"\benviron(?:ment(?:al)?)?\b",                        False),  # NEW (13 funds)
    ("green",                   "EN",    r"\bgreen\b",                                          False),  # NEW (28 funds)
    ("biodiversity",            "EN",    r"\bbiodiversit",                                      False),  # NEW — root covers EN/FR/DE variants (17 funds)
    ("circular_economy",        "EN",    r"\bcircular\b",                                       False),  # NEW (10 funds)
    ("ocean",                   "EN",    r"\bocean\b",                                          False),  # NEW (2 funds)
    ("water",                   "EN",    r"\bwater(?:fonds)?\b",                                False),  # NEW (19 funds)
    ("natural_capital",         "EN",    r"\bnatural[\s\-]capital\b",                           False),  # NEW (1 fund)
    ("natural_resources",       "EN",    r"\bnatural[\s\-]resources\b",                         False),  # NEW (2 funds)
    ("social",                  "EN",    r"\bsocial\b",                                         False),  # NEW (18 funds)
    ("inclusion",               "EN",    r"\binclusion\b",                                      False),  # NEW (3 funds)
    ("gender",                  "EN",    r"\bgender\b",                                         False),  # NEW (1 fund)
    ("diversity_theme",         "EN",    r"\bdiversity\b",                                      True),   # NEW — standalone Diversity (5 funds); review: could be statistical diversity
    ("positive_impact",         "EN",    r"\bpositive\b",                                       True),   # NEW — standalone Positive (6 funds); review: confirm impact-label intent
    ("engagement",              "EN",    r"\bengagement\b",                                     False),
    ("stewardship",             "EN",    r"\bstewards?(?:hip)?\b",                              False),
    ("governance",              "EN",    r"\bgovernance\b",                                     False),  # NEW (4 funds)
    ("4change",                 "EN",    r"\b4[Cc]hange\b",                                     False),  # NEW — R-co 4Change impact range (3 funds)
    ("calvert",                 "EN",    r"\bCalvert\b",                                        False),  # NEW — Morgan Stanley Calvert impact brand (7 funds)

    # ── French ─────────────────────────────────────────────
    ("impact_fr",               "FR",    r"\bimpact\b",                                         False),
    ("transition_fr",           "FR",    r"\btransition\b",                                     False),
    ("transformation_fr",       "FR",    r"\btransformation\b",                                 False),
    ("dev_durable",             "FR",    r"\bd[ée]veloppement[\s\-]durable\b",                  False),
    ("durable",                 "FR",    r"\bdurables?\b",                                      False),  # NEW — standalone Durable/Durables (15 funds)
    ("responsable_fr",          "FR",    r"\bresponsables?\b",                                  False),  # NEW — Responsable/Responsables (20 funds)
    ("solidaire",               "FR",    r"\bsolidaires?\b|\bsolidarit[eé]\b",                  False),  # NEW — Solidaire, Solidarité (13 funds)
    ("planete",                 "FR",    r"\bplan[eè]te\b",                                     False),  # NEW — Planète (7 funds)
    ("environnement",           "FR",    r"\benvironnement\b",                                  False),  # NEW (8 funds)
    ("carbone_fr",              "FR",    r"\bcarbone\b",                                        False),  # NEW — Carbone (6 funds)
    ("ecologique",              "FR",    r"[éÉ]cologiqu?e?\b|\b[Ee]colog",                      False),  # NEW — Écologique/Ecologique (2 funds)
    ("socialement",             "FR",    r"\bsocialement\b",                                    False),  # NEW — Socialement Responsable (1 fund)
    ("action_climatique",       "FR",    r"\baction[\s\-]climatique\b",                         False),  # full phrase — not bare "action"
    ("engagement_fr",           "FR",    r"\bengagement\b",                                     False),
    ("isr",                     "FR",    r"\bISR\b",                                            False),  # Investissement Socialement Responsable (89 funds)
    ("generations_futures",     "FR",    r"\bg[ée]n[ée]rations?[\s\-]futures?\b",               False),
    ("meilleur_monde",          "FR",    r"\bmeilleur[\s\-]monde\b",                            False),
    ("changement_positif",      "FR",    r"\bchangement[\s\-]positif\b",                        False),

    # ── German ─────────────────────────────────────────────
    ("impact_de",               "DE",    r"\bimpact\b",                                         False),
    ("transformation_de",       "DE",    r"\btransformation\b",                                 False),
    ("transition_de",           "DE",    r"\btransition\b",                                     False),
    ("nachhaltig",              "DE",    r"nachhaltig",                                         False),  # NEW — root matches Nachhaltigkeit, Nachhaltigkeitsfonds, etc. (16 funds)
    ("nachhaltige_entwicklung", "DE",    r"\bnachhaltige[\s\-]entwicklung\b",                   False),  # specific phrase kept for label precision
    ("umwelt",                  "DE",    r"\bumwelt(?:invest|fonds)?\b",                        False),  # NEW — Umwelt, UmweltInvest, Umweltfonds (5 funds)
    ("verantwortung",           "DE",    r"\bverantwortung\b",                                  False),  # NEW — Verantwortung (responsibility) (1 fund)
    ("zukunft",                 "DE",    r"\bzukunft(?:s|beweger|sstrategie)?\b",               False),  # NEW — Zukunft, Zukunftbeweger, Zukunftsstrategie (5 funds)
    ("energiewende",            "DE",    r"\benergie?wende\b",                                  False),  # NEW — German energy transition (2 funds)
    ("klima_de",                "DE",    r"\bklima\b",                                          False),  # NEW — standalone Klima; complements klimaschutz/klimaaktion
    ("klimaschutz",             "DE",    r"\bklimaschutz\b",                                    False),
    ("klimaaktion",             "DE",    r"\bklima(?:aktion|schutz)\b",                         False),
    ("oeko",                    "DE",    r"[öÖ]ko\b|[Oo]eko\b",                                 False),  # NEW — Öko/Oeko (ecological) (5 funds); trailing \b excludes mid-word matches
    ("sozial_de",               "DE",    r"\bsoziale?\b",                                       False),  # NEW — Sozial/Soziale (1 fund)
    ("wasser_de",               "DE",    r"\bwasser\b",                                         False),  # NEW — Wasser (water); \b excludes Wasserstoff (hydrogen)
    ("engagement_de",           "DE",    r"\bengagement\b",                                     False),
    ("generationen",            "DE",    r"\bgenerationen\b",                                   False),
    ("positiver_wandel",        "DE",    r"\bpositiver?[\s\-]wandel\b",                         False),
    ("bessere_welt",            "DE",    r"\bbessere[\s\-]welt\b",                              False),

    # ── Spanish ────────────────────────────────────────────
    ("impacto",                 "ES",    r"\bimpacto\b",                                        False),
    ("transicion",              "ES",    r"\btransici[oó]n\b",                                  False),
    ("transformacion",          "ES",    r"\btransformaci[oó]n\b",                              False),
    ("sostenible",              "ES",    r"\bsostenible\b|\bsostenibilidad\b",                  False),  # NEW — Sostenible, Sostenibilidad (3 funds)
    ("desarrollo_sostenible",   "ES",    r"\bdesarrollo[\s\-]sostenible\b",                     False),
    ("ambiental",               "ES",    r"\bambiental\b",                                      False),  # NEW — Ambiental (environmental)
    ("planeta_es",              "ES",    r"\bplaneta\b",                                        False),  # NEW — Planeta (planet)
    ("accion_climatica",        "ES",    r"\bacci[oó]n[\s\-]clim[aá]tica\b",                   False),  # full phrase — not bare "acción"
    ("compromiso",              "ES",    r"\bcompromiso\b",                                     True),   # "engagement" but also general; review
    ("generaciones_futuras",    "ES",    r"\bgeneraciones?[\s\-]futuras?\b",                    False),
    ("ods",                     "ES",    r"\bODS\b",                                            False),  # Objetivos de Desarrollo Sostenible
    ("mejor_mundo",             "ES",    r"\bmejor[\s\-]mundo\b",                               False),

    # ── Italian ────────────────────────────────────────────
    ("impatto",                 "IT",    r"\bimpatto\b",                                        False),
    ("transizione",             "IT",    r"\btransizione\b",                                    False),
    ("trasformazione",          "IT",    r"\btrasformazione\b",                                 False),
    ("sostenibile_it",          "IT",    r"sostenibil",                                         False),  # NEW — Sostenibile, Sostenibilità (1 fund)
    ("sviluppo_sostenibile",    "IT",    r"\bsviluppo[\s\-]sostenibile\b",                      False),
    ("etico_it",                "IT",    r"\betic[oi]\b",                                       False),  # NEW — Etico (m.), Etici (pl.) — Italian ethical
    ("ambientale_it",           "IT",    r"\bambientale\b",                                     False),  # NEW — Ambientale (environmental)
    ("azione_climatica",        "IT",    r"\bazione[\s\-]climatica\b",                          False),
    ("generazioni_future",      "IT",    r"\bgenerazioni[\s\-]future\b",                        False),

    # ── Swedish ────────────────────────────────────────────
    ("hallbar",                 "SV",    r"\bh[åä]llbar\b",                                     False),  # NEW — standalone Hållbar (sustainable) (4 funds); note: å not ä — corrects existing hallbar_utveckling pattern
    ("hallbar_utveckling",      "SV",    r"\bh[åä]llbar[\s\-]utveckling\b",                     False),  # Hållbar utveckling (sustainable development)
    ("omstallning",             "SV",    r"\bomst[åä]llning\b",                                 False),  # Omställning (transition)
    ("klimataktion",            "SV",    r"\bklimat(?:aktion|akt)\b",                           False),
    ("engagement_sv",           "SV",    r"\bengagemang\b",                                     False),
    ("framtida_generationer",   "SV",    r"\bframtida[\s\-]generationer\b",                     False),

    # ── Danish / Norwegian ─────────────────────────────────
    ("ansvarlig",               "DA/NO", r"\bansvarlige?\b",                                    False),  # NEW — Ansvarlig/Ansvarlige (responsible) (5 funds)
    ("barekraft",               "DA/NO", r"b[æa]rekraft",                                       False),  # NEW — Bærekraft (sustainable, NO) (1 fund)
    ("klima_no_dk",             "DA/NO", r"\bklima\b",                                          False),  # NEW — Klima in Nordic context
    ("baeredygtighed",          "DA/NO", r"\bb[æa]redygtighed\b",                               False),  # Bæredygtighed (sustainability, DA)
    ("overgang",                "DA/NO", r"\bovergang\b",                                       True),   # transition (also means "crossing"); review
    ("ans_tran_abbrev",         "DA/NO", r"\bAnsTran\b",                                        True),   # likely "Ansvarlig Transition"; review

    # ── Dutch ──────────────────────────────────────────────
    ("impact_nl",               "NL",    r"\bimpact\b",                                         False),
    ("duurzaam",                "NL",    r"\bduurzaams?\b",                                     False),  # NEW — Duurzaam/Duurzame (sustainable) (5 funds)
    ("duurzame_ontwikkeling",   "NL",    r"\bduurzame[\s\-]ontwikkeling\b",                     False),
    ("milieu",                  "NL",    r"\bmilieu\b",                                         False),  # NEW — Milieu (environment; also used in French) (1 fund)
    ("overgang_nl",             "NL",    r"\bovergang\b",                                       True),   # transition; review
    ("klimaatactie",            "NL",    r"\bklimaatactie\b",                                   False),
]

# Compile patterns
COMPILED_PATTERNS = [
    (label, lang, re.compile(pat, FLAGS), review)
    for label, lang, pat, review in PATTERNS
]

print(f"Loaded {len(COMPILED_PATTERNS)} patterns across "
      f"{len(set(lang for _, lang, _, _ in PATTERNS))} language groups.")


Loaded 120 patterns across 8 language groups.


## CELL 3 — Abbreviation Expansion Map

Token-level expansion applied to matched fund names only. Best-effort: expands known tokens, flags residual unknowns in `Expansion_Complete` column.

In [4]:
# Applied to matched tokens only — not full unabbreviation
# of the entire fund name (too error-prone at scale).
# ============================================================

# Token-level expansion: token → expanded form
# Tokens must match exactly (case-sensitive after stripping)
ABBREV_EXPANSION = {
    "Sust":    "Sustainable",
    "Sus":     "Sustainable",
    "Dev":     "Development",
    "Gens":    "Generations",
    "Clmt":    "Climate",
    "Clim":    "Climate",
    "Env":     "Environmental",
    "Soc":     "Social",
    "Gov":     "Governance",
    "Eq":      "Equity",          # most common meaning
    "Fds":     "Funds",
    "Fd":      "Fund",
    "Mkt":     "Market",
    "Mkts":    "Markets",
    "Intl":    "International",
    "Intern":  "International",
    "Glb":     "Global",
    "Glbl":    "Global",
    "Emg":     "Emerging",
    "Em":      "Emerging",
    "Cnsrv":   "Conservative",
    "Eqs":     "Equities",
}

# Tokens that are KNOWN negatives (Trans → Transportation)
# If the matched abbreviation appears in this context, downgrade to review
KNOWN_NEGATIVE_ABBREVS = {
    "Transp": "Transportation",
    "TransP": "Transportation",
}

def expand_name(fund_name: str) -> tuple[str, bool]:
    """
    Best-effort token expansion. Returns (expanded_name, fully_expanded).
    fully_expanded=False if any token could not be confidently expanded.
    """
    tokens = re.split(r'(\s+)', fund_name)  # preserve whitespace
    expanded_tokens = []
    fully_expanded = True
    for token in tokens:
        stripped = token.strip()
        if stripped in ABBREV_EXPANSION:
            expanded_tokens.append(ABBREV_EXPANSION[stripped])
        elif stripped in KNOWN_NEGATIVE_ABBREVS:
            expanded_tokens.append(KNOWN_NEGATIVE_ABBREVS[stripped])
        elif re.match(r'^[A-Z]{2,5}$', stripped) and stripped not in {
            "EUR", "USD", "GBP", "CHF", "JPY", "SEK", "NOK", "DKK",  # currencies
            "ETF", "UCITS", "ELTIF",                                    # fund types
            "ESG", "ISR", "SDG", "ODS", "ODD",                         # already-known acronyms
            "ACC", "INC", "CAP", "DIS",                                 # share class
            "UK", "US", "EU", "EM", "EMU",                              # geography
            "AI", "IT", "IP",                                           # tech / share class
        }:
            # Uppercase token we don't know → mark as incomplete
            expanded_tokens.append(token)
            fully_expanded = False
        else:
            expanded_tokens.append(token)
    return "".join(expanded_tokens), fully_expanded


## CELL 4 — Token Scan (exploratory)

Exploratory. Prints uppercase abbreviations in the full dataset not in the known-safe set. Run once to check for gaps in `ABBREV_EXPANSION` before committing to the full matching pass.

In [5]:
# ============================================================
# SAFE_TOKENS — Full Reference
# Tokens confirmed to have no impact-keyword meaning.
# Organized by category for maintainability.
# ============================================================

SAFE_TOKENS = {

    # ── Currencies ─────────────────────────────────────────
    "EUR",   # Euro
    "USD",   # US Dollar
    "GBP",   # British Pound
    "CHF",   # Swiss Franc
    "JPY",   # Japanese Yen
    "SEK",   # Swedish Krona
    "NOK",   # Norwegian Krone
    "DKK",   # Danish Krone
    "HKD",   # Hong Kong Dollar
    "PLN",   # Polish Zloty
    "CZK",   # Czech Koruna
    "HUF",   # Hungarian Forint

    # ── Fund legal structures ───────────────────────────────
    "ETF",   # Exchange-Traded Fund
    "UCITS", # Undertakings for Collective Investment in Transferable Securities
    "ELTIF", # European Long-Term Investment Fund
    "AIF",   # Alternative Investment Fund
    "SICAV", # Société d'Investissement à Capital Variable (open-ended investment company)
    "FCP",   # Fonds Commun de Placement (French contractual fund)
    "SICAF", # Société d'Investissement à Capital Fixe (closed-ended)
    "OEIC",  # Open-Ended Investment Company (UK equivalent of SICAV)
    "FGR",   # Fonds voor Gemene Rekening (Dutch pooled fund structure)

    # ── ESG / responsible investment labels ────────────────
    "ESG",   # Environmental, Social and Governance — now in PATTERNS
    "SRI",   # Socially Responsible Investment — confirmed in scope (Dirk, 2026-06-23); now in PATTERNS
    "ISR",   # Investissement Socialement Responsable (French SRI label) — in PATTERNS
    "SDG",   # Sustainable Development Goals — in PATTERNS
    "ODS",   # Objetivos de Desarrollo Sostenible (Spanish SDG equivalent)
    "ODD",   # Objectifs de Développement Durable (French SDG equivalent)
    "CSR",   # Corporate Social Responsibility

    # ── Share class / series identifiers ───────────────────
    # Single letters
    "A", "B", "C", "D", "E", "F", "G", "H", "I",
    "J", "K", "L", "M", "N", "P", "Q", "R", "S",
    "T", "V", "W", "X", "Y", "Z",
    # Two-letter share class codes
    "AC",    # Accumulating Class
    "AD",    # Accumulating/Distributing Class
    "BI",    # Nordea institutional share class
    "BP",    # Base Portfolio class (JOHCM, Nordea)
    "DBI",   # Degroof Beleggingsfonds Institutioneel — Belgian institutional tax class
    "DIS",   # Distributing
    "EF",    # Equity Fund (share class suffix, e.g. SWC EF)
    "FC",    # Feeder/Founders Class
    "GF",    # Growth Fund share class
    "IA",    # Institutional Accumulating
    "IC",    # Institutional Class / Income Class
    "II",    # Second institutional series
    "IM",    # Institutional Management share class  ← also used as brand prefix (AXA IM)
    "INC",   # Income
    "KL",    # Kundeklasse (Danish: customer/institutional portfolio class)
    "LC",    # Local Currency / share class
    "LD",    # Local Distributing (DWS convention)
    "MH",    # LBPAM/French institutional share class (Mandats)
    "NA",    # North America (geographic designation in fund names)
    "PC",    # Pension Class
    "RC",    # Retail Class (French/Belgian convention)
    "RDT",   # Revenus Définitivement Taxés (Belgian institutional tax regime, paired with DBI)
    "REI",   # Real Estate Income
    "RET",   # Retail / Return class
    "SI",    # Share class identifier
    "XC",    # Extended/Exchange Class
    # Three-letter series codes
    "ACC",   # Accumulating
    "CAP",   # Capitalising
    "ESR",   # Épargne Salariale Responsable (French employee savings share class, Amundi)
    "GIF",   # HSBC Global Investment Funds (platform/series)
    "GSF",   # Goldman Sachs Funds (sub-series)
    "III",   # Third fund series (e.g. BNP Paribas III)
    "ISF",   # Schroder International Selection Fund (series)
    "FAM",   # Fund platform identifier (thematic Luxembourg SICAV platform)
    "WW",    # Worldwide (Baillie Gifford WW fund range)
    # Four/five-letter series
    "INVF",  # Invesco Funds (series)
    "MDPS",  # Fund platform/vehicle identifier (paired with TOBAM)
    "AXAWF", # AXA World Funds (series)
    "BNPP",  # BNP Paribas (abbreviated brand, used as series prefix)
    "FTGF",  # Franklin Templeton Global Funds (series)
    "UBAM",  # Union Bancaire Privée AM (series)

    # ── Fund manager / brand codes ──────────────────────────
    "AB",    # AllianceBernstein
    "AAF",   # Aegon Asset Funds (platform)
    "AKL",   # SEBinvest AKL (SEB's Danish sub-brand)
    "AM",    # Asset Management (generic suffix)
    "AXA",   # AXA Investment Managers
    "AZ",    # Azimut (Italian asset manager)
    "BGF",   # BlackRock Global Funds
    "BHF",   # BHF Asset Servicing (German)
    "BL",    # Banque de Luxembourg Investments
    "CM",    # Crédit Mutuel (CM-AM = Crédit Mutuel Asset Management)
    "CPR",   # CPR Asset Management (Amundi subsidiary)
    "CT",    # Columbia Threadneedle Investments
    "DBI",   # Degroof Petercam Belgian institutional class (see share class above)
    "DNCA",  # DNCA Finance (French, Natixis subsidiary)
    "DNB",   # DNB Asset Management (Norwegian, Den Norske Bank)
    "DPAM",  # Degroof Petercam Asset Management
    "DWS",   # DWS Investment (Deutsche Bank AM)
    "ERSTE", # Erste Asset Management (Austrian)
    "FSSA",  # First Sentier Stewart Asia (formerly First State Stewart Asia)
    "GAM",   # GAM Investments (Swiss)
    "GS",    # Goldman Sachs Asset Management
    "HSBC",  # HSBC Asset Management
    "JPM",   # J.P. Morgan Asset Management
    "KBC",   # KBC Asset Management (Belgian)
    "KBI",   # KBI Global Investors (Irish, Amundi subsidiary)
    "LBPAM", # La Banque Postale Asset Management (French)
    "LCL",   # LCL (Crédit Lyonnais subsidiary, French retail bank)
    "LGT",   # LGT Capital Partners (Liechtenstein)
    "LLB",   # Liechtensteinische Landesbank
    "LO",    # Lombard Odier Investment Managers
    "MFS",   # MFS Investment Management (Massachusetts Financial Services)
    "MS",    # Morgan Stanley Investment Management
    "NT",    # Northern Trust Asset Management
    "ODIN",  # ODIN Forvaltning (Norwegian asset manager)
    "QI",    # Quantitative Investing (Robeco QI sub-brand)
    "RBC",   # Royal Bank of Canada / RBC Global Asset Management
    "SEB",   # Skandinaviska Enskilda Banken (Swedish)
    "SG",    # Société Générale Asset Management
    "SWC",   # Swiss Life Asset Managers
    "THEAM", # Theam S.A. (BNP Paribas ETF/passive unit)
    "TOBAM", # TOBAM Asset Management (To Be A Mirror, French quant)
    "UBS",   # UBS Asset Management
    "UFF",   # Union Financière de France
    "JSS",   # J. Safra Sarasin (Swiss private bank)
    "WF",    # French asset manager brand (WF Actions series)

    # ── Domicile / geography ────────────────────────────────
    "CHN",   # China
    "EU",    # European Union
    "EM",    # Emerging Markets
    "EMU",   # European Monetary Union (Eurozone)
    "ES",    # Spain / Spanish share class designation
    "IE",    # Ireland (domicile)
    "LU",    # Luxembourg (ISIN prefix / domicile)
    "LUX",   # Luxembourg (domicile, long form)
    "NL",    # Netherlands
    "UK",    # United Kingdom
    "US",    # United States
    "USA",   # United States of America

    # ── Strategy / style descriptors ───────────────────────
    "EQ",    # Equity (Goldman Sachs Dutch fund convention)
    "ETI",   # Entreprises de Taille Intermédiaire (French: mid-sized companies)
    "PME",   # Petites et Moyennes Entreprises (French: SMEs)
    "RE",    # Real Estate
    "SMID",  # Small-Mid Cap
    "STOCK", # Equities/stocks (Erste AM convention for equity funds)
    "DXL",   # Share class used in LUX-domiciled fund series
    "FGR",   # Fonds voor Gemene Rekening (Dutch: common account fund, legal structure)
    "AI",    # Artificial Intelligence (thematic) / share class
    "IP",    # Intellectual Property / share class
    "KID",   # Key Information Document (regulatory)
    "KIID",  # Key Investor Information Document (regulatory)
    "FI",    # Fondo de Inversión (Spanish: investment fund)

}

# SRI confirmed in scope by Dirk (2026-06-23).
# Added to PATTERNS as ("sri", "EN", r"\bSRI\b", False) and to SAFE_TOKENS above.

In [6]:
# Run once on the full dataset to surface unknown abbreviations
# before finalising the ABBREV_EXPANSION map.
# ============================================================

print("Loading data...")
df = pd.read_excel(INPUT_FILE)
print(f"  {len(df)} funds, {len(df.columns)} columns")

all_tokens = []
for name in df[NAME_COL].dropna():
    tokens = re.split(r'[\s\-–&/()+]+', str(name))
    all_tokens.extend(t for t in tokens if t)

token_counts = Counter(all_tokens)

unknown_abbrevs = [
    (tok, count)
    for tok, count in token_counts.most_common(500)
    if re.match(r'^[A-Z]{2,5}$', tok)
    and tok not in SAFE_TOKENS
    and count >= 2
]

print(f"\n{'Token':<12} {'Count':>6}   (possible meaning)")
print("─" * 45)
for tok, count in unknown_abbrevs[:40]:
    known = ABBREV_EXPANSION.get(tok, "?")
    print(f"{tok:<12} {count:>6}   {known}")


Loading data...
  5680 funds, 133 columns

Token         Count   (possible meaning)
─────────────────────────────────────────────
BNP              66   ?
FF               19   ?
RV               16   ?
UB               12   ?
CI               11   ?
CR               10   ?


## CELL 5 — Matching Function

Core matching logic. Returns all pattern hits for a single fund name string.

In [7]:
def match_fund(fund_name: str) -> list[dict]:
    """
    Returns a list of match records for a single fund name.
    One record per pattern matched (a name may match multiple patterns).
    """
    matches = []
    for label, lang, compiled_re, needs_review in COMPILED_PATTERNS:
        m = compiled_re.search(fund_name)
        if m:
            matched_text = m.group(0)
            matches.append({
                "matched_pattern_label": label,
                "matched_language":      lang,
                "matched_text":          matched_text,
                "needs_review":          needs_review,
            })
    return matches


## CELL 6 — Run Matching

Applies `match_fund()` across all funds. Attaches all objective and strategy text columns to each match row.

In [8]:
print("Running pattern matching...")

# Only keep objective columns that exist in this dataset
available_obj_cols = [c for c in OBJECTIVE_COLUMNS if c in df.columns]

results = []

for _, row in df.iterrows():
    fund_name = str(row.get(NAME_COL, ""))
    fund_id   = row.get(ID_COL, "")

    matches = match_fund(fund_name)
    if not matches:
        continue

    expanded_name, fully_expanded = expand_name(fund_name)

    # Collect all objective/strategy text for output columns
    obj_texts = {col: row.get(col, "") for col in available_obj_cols}

    # One row per matched pattern
    for match in matches:
        results.append({
            ID_COL:              fund_id,
            NAME_COL:            fund_name,
            "Name_Expanded":     expanded_name,
            "Expansion_Complete": fully_expanded,
            **match,
            **obj_texts,
        })

results_df = pd.DataFrame(results)
print(f"  {results_df[ID_COL].nunique()} funds matched across {len(results_df)} pattern hits")


Running pattern matching...
  944 funds matched across 1300 pattern hits


## CELL 7 — Summary Statistics

Printed console summary: total candidates, hit counts by pattern label and language group, review flag count.

In [9]:
if len(results_df) == 0:
    print("No matches found.")
else:
    # Deduplicate to one row per fund for counting
    funds_df = results_df.drop_duplicates(subset=ID_COL)

    print(f"\n{'═'*55}")
    print(f"  IMPACT FUND CANDIDATES: {len(funds_df)} of {len(df)} funds")
    print(f"  ({len(funds_df)/len(df)*100:.1f}% of full dataset)")
    print(f"{'═'*55}")

    print(f"\nHits by pattern label:")
    for label, count in results_df["matched_pattern_label"].value_counts().items():
        n_review = results_df[results_df["matched_pattern_label"] == label]["needs_review"].sum()
        flag = "  ⚠ needs review" if n_review > 0 else ""
        print(f"  {label:<30} {count:>4}{flag}")

    print(f"\nHits by language group:")
    for lang, count in results_df["matched_language"].value_counts().items():
        print(f"  {lang:<10} {count:>4}")

    review_count = results_df["needs_review"].sum()
    print(f"\nRows flagged for human review: {review_count} "
          f"({review_count/len(results_df)*100:.1f}% of all hits)")



═══════════════════════════════════════════════════════
  IMPACT FUND CANDIDATES: 944 of 5680 funds
  (16.6% of full dataset)
═══════════════════════════════════════════════════════

Hits by pattern label:
  esg                             164
  sustainable                     113
  sust_dev_abbrev                 107  ⚠ needs review
  isr                              89
  impact                           57
  impact_fr                        57
  impact_de                        57
  impact_nl                        57
  climate                          51
  sri                              45
  transition                       35
  transition_fr                    35
  transition_de                    35
  green                            28
  responsable_fr                   20
  water                            19
  social                           18
  biodiversity                     17
  sdg                              17
  nachhaltig                       16
  durable        

## CELL 8 — Output to Excel

Writes four sheets: **Matches** (one row per hit), **Funds_Deduped** (one row per fund), **Summary** (pattern-level stats), **Token_Scan** (unknown abbreviation candidates).

In [10]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
outfile = OUTPUT_DIR / f"Impact_Fund_Candidates_{timestamp}.xlsx"

with pd.ExcelWriter(outfile, engine="openpyxl") as writer:

    # Sheet 1 — Full results (one row per pattern hit)
    results_df.to_excel(writer, sheet_name="Matches", index=False)

    # Sheet 2 — One row per fund (first/most-significant match)
    if len(results_df) > 0:
        deduped = (
            results_df
            .sort_values("needs_review")          # confirmed hits first
            .drop_duplicates(subset=ID_COL, keep="first")
        )
        deduped.to_excel(writer, sheet_name="Funds_Deduped", index=False)

    # Sheet 3 — Summary statistics
    if len(results_df) > 0:
        summary_rows = []
        for label, grp in results_df.groupby("matched_pattern_label"):
            lang = grp["matched_language"].iloc[0]
            n_funds = grp[ID_COL].nunique()
            n_review = int(grp["needs_review"].sum())
            example = grp[NAME_COL].iloc[0]
            summary_rows.append({
                "Pattern Label":    label,
                "Language":         lang,
                "Funds Matched":    n_funds,
                "Needs Review (n)": n_review,
                "Example Name":     example,
            })
        summary_df = pd.DataFrame(summary_rows).sort_values("Funds Matched", ascending=False)
        summary_df.to_excel(writer, sheet_name="Summary", index=False)

    # Sheet 4 — Token scan: unknown abbreviations from Cell 4
    abbrev_rows = [
        {"Token": tok, "Count": count, "Known Expansion": ABBREV_EXPANSION.get(tok, "UNKNOWN")}
        for tok, count in unknown_abbrevs[:100]
    ]
    pd.DataFrame(abbrev_rows).to_excel(writer, sheet_name="Token_Scan", index=False)

print(f"\nOutput written to:\n  {outfile}")



Output written to:
  /Users/dannyhogan/Desktop/Hogan_RA_Work/Impact_Fund_Candidates_20260623_1120.xlsx


In [11]:
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
outfile   = OUTPUT_DIR / f"Impact_Fund_Candidates_{timestamp}.xlsx"

# ── helpers ──────────────────────────────────────────────────────────────────

def _fill(hex_color):
    return PatternFill("solid", fgColor=hex_color)

def _font(bold=False, size=10, color="000000"):
    return Font(name="Arial", bold=bold, size=size, color=color)

THIN = Border(
    left=Side(style="thin", color="D0D0D0"), right=Side(style="thin", color="D0D0D0"),
    top=Side(style="thin", color="D0D0D0"),  bottom=Side(style="thin", color="D0D0D0"),
)

def cell(ws, row, col, value="", bold=False, size=10, bg=None, fg="000000",
         halign="center", wrap=False):
    c = ws.cell(row=row, column=col, value=value)
    c.font      = _font(bold, size, fg)
    c.alignment = Alignment(horizontal=halign, vertical="center", wrap_text=wrap)
    c.border    = THIN
    if bg:
        c.fill = _fill(bg)
    return c

def mwrite(ws, r1, c1, r2, c2, value="", bold=False, size=10, bg=None, fg="000000"):
    ws.merge_cells(start_row=r1, start_column=c1, end_row=r2, end_column=c2)
    c = ws.cell(row=r1, column=c1, value=value)
    c.font      = _font(bold, size, fg)
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    c.border    = THIN
    if bg:
        c.fill = _fill(bg)
    return c

GREEN_D = "1A6B42"   # dark header
GREEN_L = "D6EFE3"   # section label row
GREEN_M = "EAF5ED"   # column headers inside sections
ALT_ROW = "F7FAF8"   # alternating row tint
BLUE_D  = "185FA5"   # Art 9
BLUE_L  = "DCE8F5"   # Art 8
AMBER   = "BA7517"   # needs review

# ── pre-compute all stats ─────────────────────────────────────────────────────

total_funds   = len(df)
matched_funds = results_df[ID_COL].nunique() if len(results_df) > 0 else 0
total_hits    = len(results_df)

if len(results_df) > 0:
    deduped = (
        results_df
        .sort_values("needs_review")
        .drop_duplicates(subset=ID_COL, keep="first")
    )
    n_review    = int(deduped["needs_review"].sum())
    n_confirmed = matched_funds - n_review

    # Summary table (existing)
    summary_rows = []
    for label, grp in results_df.groupby("matched_pattern_label"):
        summary_rows.append({
            "Pattern Label":    label,
            "Language":         grp["matched_language"].iloc[0],
            "Funds Matched":    grp[ID_COL].nunique(),
            "Needs Review (n)": int(grp["needs_review"].sum()),
            "Example Name":     grp[NAME_COL].iloc[0],
        })
    summary_df = pd.DataFrame(summary_rows).sort_values("Funds Matched", ascending=False)

    # Language totals: unique funds matched by any pattern in that language
    lang_totals = (
        results_df.groupby("matched_language")[ID_COL]
        .nunique().sort_values(ascending=False)
    )

    # SFDR counts (join from full dataset)
    sfdr_col = "EU SFDR Fund type (Article 8 or Article 9)"
    if sfdr_col in df.columns:
        matched_ids = set(results_df[ID_COL])
        sfdr = df[df[ID_COL].isin(matched_ids)][sfdr_col].value_counts()
        n_art8 = int(sfdr.get("Article 8", 0))
        n_art9 = int(sfdr.get("Article 9", 0))
    else:
        n_art8 = n_art9 = 0

    # ── Term × Language long table ────────────────────────────────────────
    # Row = one (token, language, pattern_label) combination
    # Columns: token | language | pattern_label | # funds |
    #          % within language group | % of full dataset
    tl = (
        results_df
        .groupby(["matched_text", "matched_language", "matched_pattern_label"])[ID_COL]
        .nunique().reset_index().rename(columns={ID_COL: "n_funds"})
    )
    tl["pct_within_lang"] = tl.apply(
        lambda r: round(r["n_funds"] / lang_totals.get(r["matched_language"], 1) * 100, 1),
        axis=1,
    )
    tl["pct_of_dataset"] = (tl["n_funds"] / total_funds * 100).round(2)
    tl = tl.sort_values(["matched_language", "n_funds"], ascending=[True, False]).reset_index(drop=True)

    # ── Language matrices ─────────────────────────────────────────────────
    lang_order = list(lang_totals.index)

    pivot_c = (
        results_df.groupby(["matched_text", "matched_language"])[ID_COL]
        .nunique().unstack(fill_value=0).reset_index()
        .rename(columns={"matched_text": "Token"})
    )
    for lg in lang_order:                            # ensure all lang cols present
        if lg not in pivot_c.columns:
            pivot_c[lg] = 0
    pivot_c = pivot_c[["Token"] + lang_order]
    pivot_c["Total"] = pivot_c[lang_order].sum(axis=1)
    pivot_c = pivot_c.sort_values("Total", ascending=False).reset_index(drop=True)

    # rates version: each cell = n_funds / that language's total × 100
    pivot_r = pivot_c.copy()
    for lg in lang_order:
        lt = lang_totals.get(lg, 1)
        pivot_r[lg] = (pivot_r[lg] / lt * 100).round(1)
    pivot_r = pivot_r.rename(columns={"Total": "Total (funds)"})

    # lang-total footer rows to append to both pivots
    count_footer = {"Token": "— LANGUAGE TOTAL (unique funds) —"}
    rate_footer  = {"Token": "— LANGUAGE TOTAL (unique funds) —"}
    for lg in lang_order:
        count_footer[lg] = int(lang_totals.get(lg, 0))
        rate_footer[lg]  = int(lang_totals.get(lg, 0))
    count_footer["Total"]          = matched_funds
    rate_footer["Total (funds)"]   = matched_funds

else:
    deduped     = pd.DataFrame()
    n_review    = n_confirmed = 0
    summary_df  = pd.DataFrame()
    lang_totals = pd.Series(dtype=int)
    n_art8 = n_art9 = 0
    tl = pivot_c = pivot_r = pd.DataFrame()
    lang_order = []

# ── write workbook ────────────────────────────────────────────────────────────

with pd.ExcelWriter(outfile, engine="openpyxl") as writer:

    # ── existing sheets (unchanged) ───────────────────────────────────────
    results_df.to_excel(writer, sheet_name="Matches",      index=False)
    if len(deduped):
        deduped.to_excel(writer,    sheet_name="Funds_Deduped", index=False)
    if len(summary_df):
        summary_df.to_excel(writer, sheet_name="Summary",       index=False)

    abbrev_rows = [
        {"Token": tok, "Count": cnt, "Known Expansion": ABBREV_EXPANSION.get(tok, "UNKNOWN")}
        for tok, cnt in unknown_abbrevs[:100]
    ]
    pd.DataFrame(abbrev_rows).to_excel(writer, sheet_name="Token_Scan", index=False)

    if len(results_df) == 0:
        print("No matches — stats sheets skipped.")
    else:
        wb = writer.book

        # ── Sheet: Overview Stats ─────────────────────────────────────────
        ws = wb.create_sheet("Overview Stats")

        for i, w in enumerate([22, 18, 18, 18, 18, 18, 18, 18, 18, 18], 1):
            ws.column_dimensions[get_column_letter(i)].width = w

        r = 1
        mwrite(ws, r, 1, r, 9, "IMPACT FUND SCREENER — RUN OVERVIEW",
               bold=True, size=13, bg=GREEN_D, fg="FFFFFF")
        ws.row_dimensions[r].height = 28

        # Section: top-line counts (2 rows per box: label + value)
        r += 2
        mwrite(ws, r, 1, r, 9, "OVERVIEW", bold=True, size=10, bg=GREEN_L, fg=GREEN_D)
        ws.row_dimensions[r].height = 18
        r += 1

        top_boxes = [
            ("Total funds\nin dataset",  f"{total_funds:,}"),
            ("Matched\nfunds",           f"{matched_funds:,}"),
            ("Match\nrate",              f"{matched_funds/total_funds*100:.1f}%"),
            ("Total\npattern hits",      f"{total_hits:,}"),
            ("Avg hits\nper fund",       f"{total_hits/max(matched_funds,1):.2f}"),
            ("Confirmed\nmatches",       f"{n_confirmed:,}"),
            ("Flagged for\nreview",      f"{n_review:,}"),
            ("Review\nrate",             f"{n_review/max(matched_funds,1)*100:.1f}%"),
        ]
        for ci, (lbl, val) in enumerate(top_boxes, 1):
            cell(ws, r,   ci, lbl, size=9,  bg=GREEN_M, fg="3A5A3A", wrap=True)
            cell(ws, r+1, ci, val, size=16, bg="FFFFFF", bold=True)
        ws.row_dimensions[r].height   = 30
        ws.row_dimensions[r+1].height = 32

        # Section: SFDR split
        r += 3
        mwrite(ws, r, 1, r, 9, "SFDR CLASSIFICATION", bold=True, size=10, bg=GREEN_L, fg=GREEN_D)
        ws.row_dimensions[r].height = 18
        r += 1

        sfdr_boxes = [
            ("Article 8",         f"{n_art8:,}",                          BLUE_L,   BLUE_D),
            ("Article 9",         f"{n_art9:,}",                          BLUE_D,   "FFFFFF"),
            ("Article 9\nrate",   f"{n_art9/max(n_art8+n_art9,1)*100:.1f}%", "FFFFFF", "000000"),
        ]
        for ci, (lbl, val, bg, fg) in enumerate(sfdr_boxes, 1):
            cell(ws, r,   ci, lbl, size=9,  bg=bg, fg=fg, wrap=True)
            cell(ws, r+1, ci, val, size=16, bg=bg, fg=fg, bold=True)
        ws.row_dimensions[r].height   = 28
        ws.row_dimensions[r+1].height = 32

        # Section: language breakdown
        r += 3
        mwrite(ws, r, 1, r, 9, "MATCHES BY LANGUAGE GROUP", bold=True, size=10, bg=GREEN_L, fg=GREEN_D)
        ws.row_dimensions[r].height = 18
        r += 1

        hdrs = ["Language", "Unique funds", "% of matched", "% of full dataset", "Top term", "Top term n"]
        for ci, h in enumerate(hdrs, 1):
            cell(ws, r, ci, h, bold=True, size=9, bg=GREEN_M, fg="3A5A3A")
        ws.row_dimensions[r].height = 16
        r += 1

        for i, (lang, lt) in enumerate(lang_totals.items()):
            sub = tl[tl["matched_language"] == lang]
            top = sub.sort_values("n_funds", ascending=False).iloc[0] if len(sub) else None
            bg_r = ALT_ROW if i % 2 else "FFFFFF"
            vals = [lang, lt, f"{lt/matched_funds*100:.1f}%",
                    f"{lt/total_funds*100:.2f}%",
                    top["matched_text"] if top is not None else "—",
                    int(top["n_funds"]) if top is not None else 0]
            for ci, v in enumerate(vals, 1):
                cell(ws, r, ci, v, size=10, bg=bg_r,
                     halign="left" if ci == 1 else "center")
            ws.row_dimensions[r].height = 15
            r += 1

        # Section: top 25 patterns
        r += 1
        mwrite(ws, r, 1, r, 9, "TOP 25 PATTERNS (by fund count)", bold=True, size=10, bg=GREEN_L, fg=GREEN_D)
        ws.row_dimensions[r].height = 18
        r += 1

        hdrs = ["Pattern label", "Language", "# funds", "% of matched", "% of full dataset",
                "Needs review n", "Needs review %"]
        for ci, h in enumerate(hdrs, 1):
            cell(ws, r, ci, h, bold=True, size=9, bg=GREEN_M, fg="3A5A3A")
        ws.row_dimensions[r].height = 16
        r += 1

        for i, (_, rd) in enumerate(summary_df.head(25).iterrows()):
            n    = rd["Funds Matched"]
            nrev = rd["Needs Review (n)"]
            bg_r = ALT_ROW if i % 2 else "FFFFFF"
            vals = [rd["Pattern Label"], rd["Language"], n,
                    f"{n/matched_funds*100:.1f}%",
                    f"{n/total_funds*100:.2f}%",
                    nrev,
                    f"{nrev/max(n,1)*100:.0f}%"]
            for ci, v in enumerate(vals, 1):
                cell(ws, r, ci, v, size=10, bg=bg_r,
                     halign="left" if ci <= 2 else "center")
            ws.row_dimensions[r].height = 15
            r += 1

        ws.freeze_panes = "A3"

        # ── Sheet: Term × Language (long format) ─────────────────────────
        ws2 = wb.create_sheet("Term × Language")

        tl_out = tl.rename(columns={
            "matched_text":            "Token",
            "matched_language":        "Language",
            "matched_pattern_label":   "Pattern Label",
            "n_funds":                 "# Funds",
            "pct_within_lang":         "% within language group",
            "pct_of_dataset":          "% of full dataset",
        })

        col_widths2 = [18, 12, 28, 10, 24, 18]
        for i, w in enumerate(col_widths2, 1):
            ws2.column_dimensions[get_column_letter(i)].width = w

        for ci, h in enumerate(tl_out.columns, 1):
            cell(ws2, 1, ci, h, bold=True, size=9, bg=GREEN_D, fg="FFFFFF")
        ws2.row_dimensions[1].height = 18

        for ri, row_data in enumerate(tl_out.itertuples(index=False), start=2):
            bg_r = ALT_ROW if ri % 2 else "FFFFFF"
            for ci, v in enumerate(row_data, 1):
                cell(ws2, ri, ci, v, size=10, bg=bg_r,
                     halign="left" if ci <= 3 else "center")
            ws2.row_dimensions[ri].height = 14

        ws2.freeze_panes = "A2"

        # ── Sheet: Language Matrix — counts ──────────────────────────────
        ws3 = wb.create_sheet("Language Matrix (counts)")

        pivot_c_out = pd.concat([pivot_c, pd.DataFrame([count_footer])], ignore_index=True)
        pivot_c_out.columns = [str(c) for c in pivot_c_out.columns]

        ws3.column_dimensions["A"].width = 20
        for i in range(2, len(pivot_c_out.columns) + 1):
            ws3.column_dimensions[get_column_letter(i)].width = 12

        # header
        for ci, h in enumerate(pivot_c_out.columns, 1):
            cell(ws3, 1, ci, h, bold=True, size=9, bg=GREEN_D, fg="FFFFFF")
        ws3.row_dimensions[1].height = 18

        # footer row index (last row)
        footer_row = len(pivot_c_out) + 1

        for ri, row_data in enumerate(pivot_c_out.itertuples(index=False), start=2):
            is_footer = (ri == footer_row)
            bg_r = GREEN_L if is_footer else (ALT_ROW if ri % 2 else "FFFFFF")
            for ci, v in enumerate(row_data, 1):
                cell(ws3, ri, ci, v, size=10, bg=bg_r, bold=is_footer,
                     halign="left" if ci == 1 else "center")
            ws3.row_dimensions[ri].height = 14

        ws3.freeze_panes = "B2"

        # ── Sheet: Language Matrix — rates (% within language group) ─────
        ws4 = wb.create_sheet("Language Matrix (rates %)")

        pivot_r_out = pd.concat([pivot_r, pd.DataFrame([rate_footer])], ignore_index=True)
        pivot_r_out.columns = [str(c) for c in pivot_r_out.columns]

        ws4.column_dimensions["A"].width = 20
        for i in range(2, len(pivot_r_out.columns) + 1):
            ws4.column_dimensions[get_column_letter(i)].width = 14

        for ci, h in enumerate(pivot_r_out.columns, 1):
            cell(ws4, 1, ci, h, bold=True, size=9, bg=GREEN_D, fg="FFFFFF")
        ws4.row_dimensions[1].height = 18

        note_col = len(pivot_r_out.columns) + 2
        ws4.cell(row=1, column=note_col, value="Note: % = funds matched by this token ÷ total funds matched by any pattern in that language group")
        ws4.cell(row=1, column=note_col).font = Font(name="Arial", size=8, italic=True, color="888888")

        footer_row = len(pivot_r_out) + 1

        for ri, row_data in enumerate(pivot_r_out.itertuples(index=False), start=2):
            is_footer = (ri == footer_row)
            bg_r = GREEN_L if is_footer else (ALT_ROW if ri % 2 else "FFFFFF")
            for ci, v in enumerate(row_data, 1):
                cell(ws4, ri, ci, v, size=10, bg=bg_r, bold=is_footer,
                     halign="left" if ci == 1 else "center")
            ws4.row_dimensions[ri].height = 14

        ws4.freeze_panes = "B2"

print(f"\nOutput written to:\n  {outfile}")
print(f"\nSheets: Matches | Funds_Deduped | Summary | Token_Scan |")
print(f"        Overview Stats | Term × Language |")
print(f"        Language Matrix (counts) | Language Matrix (rates %)")



Output written to:
  /Users/dannyhogan/Desktop/Hogan_RA_Work/Impact_Fund_Candidates_20260623_1323.xlsx

Sheets: Matches | Funds_Deduped | Summary | Token_Scan |
        Overview Stats | Term × Language |
        Language Matrix (counts) | Language Matrix (rates %)
